# Experiment C: Final Ensemble (The Winner's Strategy)

이 노트북은 프로젝트의 최종 완성판으로, 다음과 같은 고도화된 전략을 결합합니다:
1. **Advanced FE:** 시계열 특성(Rolling/Lag) 및 물리적 마모 지표 반영
2. **Target Encoding:** 드라이버의 성향을 수치화하여 정보량 극대화
3. **Multi-Model Ensemble:** CatBoost, LightGBM, XGBoost 3대 부스팅 모델 활용
4. **Rank Averaging:** AUC 지표 최적화를 위한 순위 기반 앙상블

## 1. 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from scipy.stats import rankdata
import gc
import warnings

warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/playground-series-s6e5/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train Shape: {train.shape}, Test Shape: {test.shape}')

## 2. 통합 특성 공학 (Feature Engineering)
EDA에서 발견한 물리적 한계점과 시계열적 흐름을 모두 반영하는 최강의 변수 세트를 생성합니다.

In [ ]:
def final_engineering(df):
    # A. 물리적 지표
    compound_mean_life = {'HARD': 25, 'MEDIUM': 18, 'SOFT': 12, 'INTERMEDIATE': 20, 'WET': 15}
    df['Relative_TyreLife'] = df['TyreLife'] / df['Compound'].map(compound_mean_life).fillna(20)
    df['Is_Final_Laps'] = (df['RaceProgress'] > 0.85).astype(int)
    df['Degradation_Momentum'] = df['TyreLife'] * df['Cumulative_Degradation']
    df['Degradation_per_Lap'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)
    
    # B. 시계열 흐름 (Rolling & Lag)
    df = df.sort_values(by=['Race', 'Driver', 'LapNumber'])
    df['LapTime_Delta_Lag1'] = df.groupby(['Race', 'Driver'])['LapTime_Delta'].shift(1).fillna(0)
    df['Rolling_Mean_Delta_3'] = df.groupby(['Race', 'Driver'])['LapTime_Delta'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    
    return df.sort_index()

train = final_engineering(train)
test = final_engineering(test)

train_enc = train.copy()
test_enc = test.copy()
cat_features = ['Driver', 'Compound', 'Race', 'Year']
le = LabelEncoder()

for col in cat_features:
    train_enc[col] = le.fit_transform(train[col].astype(str))
    test_enc[col] = le.transform(test[col].astype(str))

drop_cols = ['id', 'PitNextLap']
features = [c for c in train.columns if c not in drop_cols]
print("Feature Engineering Complete.")

## 3. 모델 정의 및 훈련 설정
성격이 다른 3가지 부스팅 모델을 정의합니다. 각 모델은 데이터의 서로 다른 패턴을 학습합니다.

In [ ]:
X = train_enc[features]
y = train['PitNextLap']
groups = train['Race']

kf = GroupKFold(n_splits=5)
oof_dict = {}
test_dict = {}

models = {
    'CatBoost': CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=6, eval_metric='AUC', verbose=0, random_seed=42, early_stopping_rounds=50),
    'LightGBM': LGBMClassifier(n_estimators=1000, learning_rate=0.03, importance_type='gain', random_state=42, verbose=-1, early_stopping_rounds=50),
    'XGBoost': XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=6, eval_metric='auc', random_state=42, early_stopping_rounds=50)
}

## 4. Multi-Model 교차 검증 및 Target Encoding
모든 모델에 대해 Fold별로 Target Encoding을 적용하며 학습을 진행합니다.

In [ ]:
for name, model in models.items():
    print(f"--- Training {name} ---")
    oof_preds = np.zeros(len(train))
    test_preds = np.zeros(len(test))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y, groups)):
        X_tr, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        X_te = test_enc[features].copy()
        
        te = TargetEncoder(cols=['Driver'], smoothing=10)
        X_tr['Driver_TE'] = te.fit_transform(X_tr['Driver'], y_tr)
        X_val['Driver_TE'] = te.transform(X_val['Driver'])
        X_te['Driver_TE'] = te.transform(X_te['Driver'])
        
        if name == 'CatBoost':
            model.fit(X_tr, y_tr, cat_features=cat_features, eval_set=(X_val, y_val))
        else:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
            
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_te)[:, 1] / 5
        
    oof_dict[name] = oof_preds
    test_dict[name] = test_preds
    print(f"{name} OOF AUC: {roc_auc_score(y, oof_preds):.4f}")

## 5. Rank Ensemble & Final Submission
각 모델이 출력한 확률의 순위(Rank)를 평균 내어 최종 예측치를 산출합니다.

In [ ]:
print("\n--- Final Rank Ensemble ---")
final_oof = (
    rankdata(oof_dict['CatBoost']) * 0.4 +
    rankdata(oof_dict['LightGBM']) * 0.3 +
    rankdata(oof_dict['XGBoost']) * 0.3
)

final_test = (
    rankdata(test_dict['CatBoost']) * 0.4 +
    rankdata(test_dict['LightGBM']) * 0.3 +
    rankdata(test_dict['XGBoost']) * 0.3
)

print(f"Ensemble OOF AUC: {roc_auc_score(y, final_oof):.4f}")

submission['PitNextLap'] = final_test
submission.to_csv('submission_final_ensemble.csv', index=False)
print("Final submission file saved.")